##### Copyright 2026 Google LLC.

In [1]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini Agents API: Build managed agents with the Interactions API

<a class="tfo-notebook-buttons" target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_managed_agents.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The [Interactions API](https://ai.google.dev/gemini-api/docs/interactions) provides a unified interface for working with Gemini models and agents. The [Getting Started notebook](./Get_started_interactions_api.ipynb) covers how to use it with standard Gemini **models** for text generation, multi-turn conversations, and tool use.

This notebook focuses on something different: **managed agents** with the `antigravity-preview-05-2026` agent.

### `agent=` vs `model=`

When you call the Interactions API, you choose between two modes:

| Parameter | What runs | Best for |
|-----------|-----------|----------|
| `model="gemini-..."` | A standard Gemini model | Text generation, structured output, function calling |
| `agent="antigravity-preview-05-2026"` | A **managed agent** in a sandboxed Linux environment | Autonomous tasks: code execution, web research, file management |

With `model=`, you get a stateless LLM call (see the [Getting Started notebook](./Get_started_interactions_api.ipynb)). With `agent=`, you spin up an autonomous agent that can **reason, plan, write and execute code, browse the web, and manage files** — all inside a secure sandbox, without you writing any orchestration logic.

This notebook walks you through the agent mode step by step:

1. **Simple questions** — use the agent like an LLM (it works, but it's overkill!)
2. **Multi-turn conversations** — persistent sandbox = built-in memory
3. **Using tools** — code execution, web search, file operations
4. **Loading data into the sandbox** — inject files before the agent starts
5. **Creating reusable custom agents** — bundle instructions, skills, and environment

<a name="setup"></a>
## Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai). It's recommended to always use the latest version.

In [1]:
%pip install -U -q "google-genai>=2.9.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 24.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.


### Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key or you aren't sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for an example.

In [2]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### Initialize SDK client

With the new SDK, now you only need to initialize a client with you API key.

In [3]:
import uuid
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=GEMINI_API_KEY)

# The default managed agent.
AGENT = "antigravity-preview-05-2026"

# Generate a unique suffix for this notebook session to prevent agent ID conflicts
UNIQUE_SUFFIX = uuid.uuid4().hex[:8]

print("Client ready!")

Client ready!


## 1. Simple questions — the agent as an LLM

The simplest way to use a managed agent is to ask it a question, just like you'd call a standard Gemini model. Pass `agent="antigravity-preview-05-2026"` and `environment="remote"` to create a fresh Linux sandbox for the agent.

This works, but it's a bit like driving a Formula 1 car to the grocery store — the agent has code execution, web search, and file management capabilities that are all sitting idle for a simple factual question.

In [4]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What is the capital of France?",
    environment="remote",
)

Markdown(interaction.output_text)

The capital of France is Paris.

In [5]:
# The response also includes metadata about the agent's sandbox.
print(f"Status:         {interaction.status}")
print(f"Interaction ID: {interaction.id}")
print(f"Environment ID: {interaction.environment_id}")

Status:         completed
Interaction ID: v1_ChdDbS1hYXJTWE05Qy1xdHNQOU1PeG9RVRIXQ20tYWFyU1hNOUMtcXRzUDlNT3hvUVU
Environment ID: 1ec6e4dd19d9b024fa568d29c6b06ffa


Notice the `environment_id` in the response. That's the agent's persistent Linux sandbox. Even for this simple question, a full container was provisioned. Let's make use of that persistence next.

## 2. Multi-turn conversations

Since each agent runs in a persistent sandbox, you can **continue where you left off** by reusing the `environment_id` and linking turns with `previous_interaction_id`.

This is fundamentally different from stateless `model=` calls. The agent has a true *persistent environment* — files it creates stick around, packages it installs remain available, and conversation context is preserved.

In [6]:
# Turn 1: Introduce yourself.
turn1 = client.interactions.create(
    agent=AGENT,
    input="Hi! My name is Alice and I'm a software engineer. Remember that in a knowledge.md doc.",
    environment= "remote",
)

Markdown(f"**Turn 1:** {turn1.output_text}")

**Turn 1:** Nice to meet you, Alice! I have recorded your name and role in `knowledge.md`. How can I assist you today?

In [7]:
# Turn 2: Ask whether the agent remembers.
# Pass environment_id and previous_interaction_id to continue the conversation.
turn2 = client.interactions.create(
    agent=AGENT,
    input="what's my name and what do I do?",
    environment= turn1.environment_id,
)

Markdown(f"**Turn 2:** {turn2.output_text}")

**Turn 2:** I don't have access to your personal identity, so I don't know your name or what you do. Each conversation here starts fresh without personal details about who you are. 

(If you were asking about the sandbox environment itself, the environment typically runs commands as `root`.)

Feel free to tell me what you're working on or what project you'd like help with today!

The agent remembered across turns because you passed `environment` with the previous environment ID — same sandbox, which means same files.

You could have achieved the same result using `previous_interaction_id` to keep the history of the previous conversation, but that would not have showcased the environement specificities.

This is how you build stateful, multi-turn workflows. See the [Getting Started notebook](./Get_started_interactions_api.ipynb) for `model=`-based multi-turn using `previous_interaction_id` alone (without environments).

## 3. Using tools — where the agent shines

This is where managed agents go beyond a standard chat model. The antigravity-preview-05-2026 agent has **built-in tools** it uses autonomously — you don't declare them, just describe your goal and the agent figures out what to use.

| Tool | Description |
|------|-------------|
| `bash` | Execute shell commands in the sandbox |
| `google_search` | Search the web for current information |
| `url_context` | Fetch and extract text from URLs |
| `write_file` | Create or overwrite files in the sandbox |
| `read_file` | Read file contents from the sandbox |
| `list_files` | List directory contents |
| `delete_file` | Remove files from the sandbox |

For the standard `model=`-based tools (Google Search grounding, code execution, function calling), see the [Getting Started notebook](./Get_started_interactions_api.ipynb) and the dedicated tool notebooks:
- [Code Execution](./Code_Execution.ipynb)
- [Search Grounding](./Search_Grounding.ipynb)
- [Function Calling](./Function_calling.ipynb)

### Code execution

Ask a computational question and the agent will write code, run it in its sandbox, and return the verified result.

In [8]:
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Write a Python script that computes the first 20 Fibonacci numbers. "
        "Run it and show the output."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

I have created and executed the Python script `fibonacci.py`.

### Script (`fibonacci.py`)

```python
def generate_fibonacci(n: int) -> list[int]:
    """Generate the first n Fibonacci numbers."""
    if n <= 0:
        return []
    if n == 1:
        return [0]
    
    sequence = [0, 1]
    for _ in range(2, n):
        sequence.append(sequence[-1] + sequence[-2])
    return sequence

if __name__ == "__main__":
    fib_numbers = generate_fibonacci(20)
    print("First 20 Fibonacci numbers:")
    print(fib_numbers)
```

### Output

```
First 20 Fibonacci numbers:
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181]
```

### Inspecting steps — what the agent actually did

The `steps` field in the response shows the agent's reasoning chain: its thoughts, tool calls, tool results, and final output. This is useful for debugging and understanding the agent's behavior.

In [10]:
# Inspect the steps from the Fibonacci interaction above.
for i, step in enumerate(interaction.steps):
    step_type = step.type
    print(f"--- Step {i} [{step_type}] ---")

    # Tool call steps show which tool was invoked and with what arguments.
    if hasattr(step, "name") and step.name:
        print(f"  Tool: {step.name}")
        if hasattr(step, "arguments"):
            args_str = str(step.arguments)[:300]
            print(f"  Args: {args_str}")

    # Content steps contain the agent's text output.
    if hasattr(step, "content") and step.content:
        for c in step.content:
            if hasattr(c, "text"):
                print(f"  Text: {c.text[:300]}")
    print()

--- Step 0 [thought] ---

--- Step 1 [code_execution_call] ---

--- Step 2 [code_execution_result] ---

--- Step 3 [function_call] ---
  Tool: write_file
  Args: {'explanation': 'Saved the Fibonacci script to fibonacci.py', 'toolAction': 'Writing file', 'toolSummary': 'Save fibonacci.py', 'path': 'fibonacci.py', 'content': 'def generate_fibonacci(n: int) -> list[int]:\n    """Generate the first n Fibonacci numbers."""\n    if n <= 0:\n        return []\n    

--- Step 4 [function_result] ---
  Tool: write_file

--- Step 5 [code_execution_call] ---

--- Step 6 [code_execution_result] ---

--- Step 7 [model_output] ---
  Text: I have created and executed the Python script `fibonacci.py`.

### Script (`fibonacci.py`)

```python
def generate_fibonacci(n: int) -> list[int]:
    """Generate the first n Fibonacci numbers."""
    if n <= 0:
        return []
    if n == 1:
        return [0]
    
    sequence = [0, 1]
    for _



### Web search

The agent can search the web autonomously when it needs up-to-date information.

In [11]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What were the top 3 news stories about Google this week? Summarize them briefly.",
    environment="remote",
)

Markdown(interaction.output_text)

Here are the top three news stories about Google this week:

### 1. Federal Judge Spares Google from Ad-Tech Breakup
In a major antitrust ruling, U.S. District Judge Leonie Brinkema rejected the Department of Justice's bid to force Google to divest key parts of its advertising business, specifically its AdX ad exchange and DoubleClick auction logic [1]. While the court affirmed that Google had engaged in anticompetitive practices to maintain a monopoly in digital advertising technology, the judge declined structural breakup remedies in favor of behavioral requirements and system retooling [1, 2]. 

### 2. Launch of Gemini 3.8 Flash and 3.8 Flash Cyber
Google expanded its AI portfolio with the launch of **Gemini 3.8 Flash** and **Gemini 3.8 Flash Cyber** [3]. Designed as a high-speed, cost-effective workhorse model priced at $0.75 per million input tokens, Gemini 3.8 Flash brings notable enhancements to autonomous agent workflows, long-horizon coding, and complex reasoning [3, 4]. The specialized Cyber variant introduces frontier-level vulnerability discovery and automated patching capabilities, made available to verified security defenders through Google's Fairwind Program [3].

### 3. Rollout of the September 2026 Android Feature Drop
Google announced and began rolling out its latest **Android Drop**, introducing five major features across supported devices [5, 6]:
* **Find Hub & Gemini Item Tracking:** Users can instruct Gemini to remember the location of non-tagged items (e.g., passports or spare keys) with optional photos [5].
* **Motion Assist:** A new screen-overlay feature in Android 17 that uses responsive visual cues to help reduce motion sickness while reading in moving vehicles [5].
* **Guided Vision:** Real-time spoken guidance and camera framing prompts within Gemini Live to assist users with low vision [6].
* **Google Messages Enhancements:** Native integration of Google Keep lists inside chat threads and new custom chat themes [5, 6].

---

### Sources
* [1] [AP News: Judge orders changes to Google's digital ads business but spares it from a breakup](https://apnews.com/article/google-advertising-technology-monopoly-penalties-d294d31fee27c45b14d5ce2196cdd80a)
* [2] [Courthouse News Service: Google dodges antitrust breakup of ad tech business](https://www.courthousenews.com/google-dodges-antitrust-breakup-of-ad-tech-business/)
* [3] [Google Blog: Introducing Gemini 3.8 Flash and 3.8 Flash Cyber](https://blog.google/innovation-and-ai/models-and-research/gemini-models/3-8-flash-and-3-8-flash-cyber/)
* [4] [Vellum: Google Gemini 3.8 Flash & 3.8 Flash Cyber Benchmarks Explained](https://www.vellum.ai/blog/gemini-3-8-flash-benchmarks-explained)
* [5] [Google Blog: September Android Drop: Remember where you put things, ease motion sickness, and more](https://blog.google/products-and-platforms/platforms/android/android-drop-september-2026/)
* [6] [Android Authority: 5 new features coming to your phone with the September 2026 Android Drop](https://www.androidauthority.com/september-2026-android-drop-3705528/)

### File operations

The agent can create, read, and manage files in its sandbox. Files persist within the environment across turns.

In [12]:
# Ask the agent to create a file, run it, and show results.
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Create a Python file called 'analysis.py' that generates 50 random numbers, "
        "computes mean, median, and standard deviation, then prints the results. "
        "Run it and show the output."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

I have created `analysis.py` and executed it.

### Code (`analysis.py`)

```python
import random
import statistics

# Generate 50 random numbers
numbers = [random.uniform(1, 100) for _ in range(50)]

# Calculate statistics
mean_val = statistics.mean(numbers)
median_val = statistics.median(numbers)
stdev_val = statistics.stdev(numbers)

# Print results
print("Generated 50 random numbers:")
print([round(n, 2) for n in numbers])
print()
print(f"Mean: {mean_val:.4f}")
print(f"Median: {median_val:.4f}")
print(f"Standard Deviation: {stdev_val:.4f}")
```

### Execution Output

```
Generated 50 random numbers:
[33.26, 29.62, 21.97, 60.69, 81.85, 8.94, 44.35, 6.2, 90.06, 70.07, 28.58, 42.57, 45.56, 16.23, 94.5, 16.71, 32.85, 75.66, 13.29, 81.9, 66.11, 10.0, 46.05, 81.63, 17.93, 72.88, 70.92, 47.51, 78.77, 32.54, 28.84, 41.19, 54.13, 11.82, 18.33, 22.68, 10.86, 61.35, 49.48, 31.82, 60.67, 57.67, 17.09, 27.77, 43.01, 53.24, 32.03, 21.14, 68.22, 62.13]

Mean: 43.8535
Median: 42.7893
Standard Deviation: 24.5919
```

## 4. Loading data into the agent's sandbox

You can inject files into the agent's environment **before it starts** using `sources`. This is how you provide data, configuration, or code for the agent to work with.

| Source type | Description | Best for |
|------------|-------------|----------|
| `inline` | Embed content directly (max 75 KB) | Config files, small scripts |
| `gcs` | Load from Google Cloud Storage | Large datasets |
| `repository` | Load from GitHub | Code repositories |

In [13]:
# Inject a CSV file inline and ask the agent to analyze it.
csv_data = """name,age,city,score
              Alice,28,Paris,92
              Bob,35,London,87
              Charlie,42,Berlin,95
              Diana,31,Tokyo,88
              Eve,26,Sydney,91"""

interaction = client.interactions.create(
    agent=AGENT,
    input="Read the file data.csv, analyze it, and tell me who scored the highest.",
    environment={
        "type": "remote",
        "sources": [
            {
                "type": "inline",
                "content": csv_data,
                "target": "/workspace/data.csv",
            }
        ],
    },
)

Markdown(interaction.output_text)

Based on the analysis of `data.csv`, **Charlie** scored the highest.

### Breakdown of Scores:
| Name | Age | City | Score |
| :--- | :--- | :--- | :--- |
| **Charlie** | **42** | **Berlin** | **95** |
| Alice | 28 | Paris | 92 |
| Eve | 26 | Sydney | 91 |
| Diana | 31 | Tokyo | 88 |
| Bob | 35 | London | 87 |

**Charlie** has the highest score with **95** points.

You can also load from other sources:

```python
# From Google Cloud Storage
{"type": "gcs", "source": "gs://my-bucket/data/", "target": "/workspace/data/"}

# From a GitHub repository
{"type": "repository", "source": "https://github.com/user/repo", "target": "/workspace/repo/"}
```

You can combine multiple sources in a single request — the agent will have access to all of them at startup.

**Pro tip:** You can use that to add skills to you agent, as you'll see next.

## 5. Creating reusable custom agents

So far, every interaction has used the base `antigravity-preview-05-2026` agent with inline instructions. Once you've found a setup that works well, you can **persist it into a named custom agent** that bundles:

- **Instructions** — system prompt that defines the agent's behavior
- **Environment** — pre-configured sandbox with files and sources
- **Skills** — `SKILL.md` files that teach the agent specialized capabilities

This is the recommended workflow:
1. **Prototype** with `agent="antigravity-preview-05-2026"` — iterate on instructions, sources, and prompts
2. **Create** a named agent via the `/agents` endpoint
3. **Invoke** your agent by name from any client

### Creating a custom agent

In [14]:
# Create a custom data analysis agent using the SDK.
my_agent = client.agents.create(
    id=f"my-data-analyst-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction=(
        "You are a data analysis assistant. "
        "Always write Python code using pandas to answer questions. "
        "Show your code and output clearly. "
        "When creating visualizations, save them as PNG files."
    ),
    base_environment={
        "type": "remote",
    },
)

print(f"✓ Agent created: {my_agent.id}")

/tmp/ipykernel_2184/1261841246.py:2: UserWarning: Agents usage is experimental and may change in future versions.
  my_agent = client.agents.create(


✓ Agent created: my-data-analyst-161cedde


### Using a custom agent

Once created, invoke your agent by name. It will follow its instructions automatically.

In [15]:
# Invoke the custom agent.
interaction = client.interactions.create(
    agent=f"my-data-analyst-{UNIQUE_SUFFIX}",
    input=(
        "Generate a sample dataset of 100 sales records with columns: "
        "product, region, revenue, quantity. "
        "Find the top 5 products by total revenue and show the analysis."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

Here is the Python code using **pandas** to generate the sample dataset, identify the top 5 products by total revenue, and produce the analysis and visualization.

---

### Python Code

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Reproducibility setup
np.random.seed(42)

# 2. Generate a 100-record sales dataset
n_records = 100
products = [
    "Laptop", "Smartphone", "Tablet", "Monitor", "Headphones",
    "Smartwatch", "Keyboard", "Mouse", "Desk Lamp", "External SSD"
]
regions = ["North", "South", "East", "West", "Central"]

base_prices = {
    "Laptop": 1200.0,
    "Smartphone": 800.0,
    "Tablet": 450.0,
    "Monitor": 300.0,
    "Smartwatch": 250.0,
    "Headphones": 150.0,
    "External SSD": 120.0,
    "Keyboard": 75.0,
    "Desk Lamp": 45.0,
    "Mouse": 30.0
}

sample_products = np.random.choice(products, size=n_records)
sample_regions = np.random.choice(regions, size=n_records)
sample_quantities = np.random.randint(1, 20, size=n_records)

# Simulate unit price variations (+/- 10%) and calculate revenue
multipliers = np.random.uniform(0.9, 1.1, size=n_records)
unit_prices = np.array([base_prices[p] for p in sample_products]) * multipliers
sample_revenues = np.round(unit_prices * sample_quantities, 2)

# Create DataFrame with required columns
df = pd.DataFrame({
    "product": sample_products,
    "region": sample_regions,
    "revenue": sample_revenues,
    "quantity": sample_quantities
})

# Save dataset to CSV
df.to_csv("sales_data.csv", index=False)

# 3. Analyze Top 5 Products by Revenue
product_summary = df.groupby("product").agg(
    total_revenue=("revenue", "sum"),
    total_quantity=("quantity", "sum"),
    transaction_count=("product", "count"),
    avg_order_value=("revenue", "mean")
).reset_index()

top_5_products = (
    product_summary
    .sort_values(by="total_revenue", ascending=False)
    .head(5)
    .reset_index(drop=True)
)

# 4. Regional breakdown for top 5 products
top_5_names = top_5_products["product"].tolist()
top_5_df = df[df["product"].isin(top_5_names)]
region_pivot = top_5_df.pivot_table(
    index="product",
    columns="region",
    values="revenue",
    aggfunc="sum",
    fill_value=0
).loc[top_5_names]

# 5. Visualization
plt.figure(figsize=(9, 5.5))
bars = plt.bar(
    top_5_products["product"], 
    top_5_products["total_revenue"], 
    color="#1f77b4", 
    edgecolor="#0d47a1", 
    width=0.6
)

plt.title("Top 5 Products by Total Revenue", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Product", fontsize=11, labelpad=8)
plt.ylabel("Total Revenue (USD)", fontsize=11, labelpad=8)
plt.grid(axis="y", linestyle="--", alpha=0.6)

for bar in bars:
    height = bar.get_height()
    plt.annotate(f"${height:,.2f}",
                 xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 5),
                 textcoords="offset points",
                 ha='center', va='bottom', fontsize=9, fontweight="bold")

plt.ylim(0, top_5_products["total_revenue"].max() * 1.12)
plt.tight_layout()
plt.savefig("top_5_products_revenue.png", dpi=300)
plt.close()
```

---

### Output & Results

#### 1. First 5 Records of the Dataset (`sales_data.csv`)
```text
      product   region  revenue  quantity
0    Keyboard     West  1120.68        16
1     Monitor    South  4854.53        16
2       Mouse    North    32.62         1
3  Headphones  Central  1402.93         9
4    Keyboard     East   456.31         6
```

#### 2. Top 5 Products by Total Revenue
```text
     product total_revenue  total_quantity  transaction_count avg_order_value
      Laptop    $99,525.64              80                  7      $14,217.95
  Smartphone    $84,399.20             104                 10       $8,439.92
      Tablet    $33,811.10              75                  9       $3,756.79
     Monitor    $32,643.40             109                  9       $3,627.04
External SSD    $13,184.28             110                 11       $1,198.57
```

#### 3. Regional Revenue Breakdown for Top 5 Products
```text
region           Central        East       North      South        West
product                                                                
Laptop             $0.00  $32,295.40  $19,482.31  $3,683.41  $44,064.52
Smartphone         $0.00  $12,566.92  $37,572.29  $5,826.15  $28,433.84
Tablet         $9,596.88   $5,465.59  $16,103.44      $0.00   $2,645.19
Monitor       $16,315.42   $5,417.60   $5,150.21  $4,854.53     $905.64
External SSD   $2,199.12     $887.46   $4,793.15    $897.27   $4,407.28
```

---

### Key Analytical Insights

1. **Revenue Concentration**:
   - **Total Overall Revenue**: **$300,115.54** across 100 transactions and 919 total units sold.
   - The top 5 products generated **$263,563.62**, accounting for **87.8%** of total revenue.
2. **Top Performer**:
   - **Laptop** ranked #1 with **$99,525.64** (33.2% of total store revenue) despite only 7 transactions, driven by a high average order value of **$14,217.95**.
3. **Volume vs. Price Driver**:
   - Products like **External SSD** and **Monitor** had high unit volumes (110 and 109 units respectively), but high-ticket items (**Laptop** and **Smartphone**) dominate overall revenue.
4. **Regional Trends**:
   - The **West** region contributed the highest revenue for Laptops ($44,064.52), while the **North** region led Smartphone sales ($37,572.29).

### Saved Artifacts
- Dataset: `sales_data.csv`
- Chart: `top_5_products_revenue.png`

### Creating an agent with pre-loaded data

You can define the agent's environment with sources, so data is ready before the agent starts:

In [16]:
# Create an agent with GCS sources pre-loaded using the SDK.
my_slides_agent = client.agents.create(
    id=f"my-gemini-api-agent-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction=(
        "You are a software engineer speciliazed in the Gemini API. "
        "Use the skills available in /.agents/skills/ to create amazing apps."
    ),
    base_environment={
        "type": "remote",
        "sources": [
            {
                "type": "repository",
                "source": "https://github.com/google-gemini/gemini-skills",
                "target": "/.agents/skills",
            }
        ],
    },
)

print(f"✓ Agent created: {my_slides_agent.id}")

✓ Agent created: my-gemini-api-agent-161cedde


In [19]:
# Invoke the custom agent.
interaction = client.interactions.create(
    agent=f"my-gemini-api-agent-161cedde",
    input="Tell me what you can do with your skills?",
    environment="remote",
)

Markdown(interaction.output_text)

With the specialized skills available in the environment, I can design and build cutting-edge applications using the **Interactions API**, **Gemini Live API**, and **Gemini Omni Flash API** across Python (`google-genai`) and TypeScript (`@google/genai`).

---

### 1. Core Gemini API Development (`gemini-api-dev`)
For building agentic workflows, complex multimodal analysis, generative media, and structured workflows using the latest Gemini models (`gemini-3.8-flash`, `gemini-3.1-pro-preview`, `gemini-3.5-flash-lite`, etc.):

- **Autonomous & Managed Agents**:
  - **Antigravity Agent (`antigravity-preview-05-2026`)**: Spin up sandboxed remote Linux environments capable of running code (Bash, Python, Node.js), managing workspaces, browsing the web, and executing multi-step software tasks.
  - **Custom Agents (`client.agents.create`)**: Build specialized persistent agents preloaded with repository sources, tools, and system instructions.
  - **Deep Research Agents (`deep-research-preview-04-2026` / `max`)**: Run background research pipelines that synthesize citations, web content, and internal files.
- **Interactions & Stateful Workflows**:
  - Stateful multi-turn interactions using server-managed context (`previous_interaction_id`).
  - Structured event streaming (`step.start`, `step.delta`, `step.stop`, typed content chunks).
- **Tools & Grounding**:
  - Synchronous and multi-tool function calling.
  - Google Search grounding, Maps grounding, URL context fetching, Code Execution, and Model Context Protocol (MCP) integrations.
- **Structured Outputs & Media Generation**:
  - Enforce strict JSON schemas / Pydantic models with thought signatures.
  - Image generation and editing (`gemini-3-pro-image` / `gemini-3.1-flash-image`).
  - Expressive speech synthesis (`gemini-3.1-flash-tts-preview`) and fast transcription (`gemini-3.5-transcribe`).
  - Multimodal semantic search and retrieval via `gemini-embedding-2`.

---

### 2. Real-Time Bidirectional Streaming (`gemini-live-api-dev`)
For low-latency, real-time voice, video, and screen-sharing applications over WebSockets:

- **Live Multimodal Streaming**:
  - Bidirectional mic-to-speaker voice streaming with `gemini-3.1-flash-live-preview`.
  - Stream live video frames (webcam or screen share) alongside raw 16kHz PCM audio for real-time visual reasoning.
- **Native Voice Activity Detection (VAD) & Interruptions**:
  - Natural interruption handling (stopping playback and clearing audio buffers immediately when the user speaks).
  - Native audio thinking with configurable `thinkingLevel`.
- **Live Translation & Transcription**:
  - Real-time speech-to-text with `gemini-3.5-transcribe-live` (interim hypotheses, verbatim or smart formatting, and Hybrid VAD).
  - Real-time speech translation across 70+ languages with `gemini-3.5-live-translate-preview`.
- **Production Session Management**:
  - Context window compression for extended conversations.
  - Session resumption and graceful handling of network disconnects (`GoAway` signals).
  - Minting **ephemeral tokens** for secure client-side browser and mobile architectures without exposing backend API keys.
  - Integrations with WebRTC/WebSocket frameworks (LiveKit, Pipecat, Fishjam, Vision Agents, Firebase AI SDK).

---

### 3. Generative Video Creation & Editing (`gemini-omni-flash-api`)
For advanced video generation, stylistic editing, and scene extension using `gemini-omni-1.1-flash`:

- **Text-to-Video & Camera Control**:
  - Generate videos from text prompts with resolutions up to **4K** (`360p`, `720p`, `1080p`, `4k`) and custom aspect ratios (`16:9` and `9:16`).
- **Image-to-Video & Transitions**:
  - Animate static images from a starting frame (`--first-frame`).
  - Interpolate smoothly between two images (`--first-frame` to `--last-frame`) to create seamless morphs, timelapses, or perfect continuous loops.
- **Multimodal Reference Guiding**:
  - Guide character appearance, objects, or artistic aesthetics using reference images (`<IMAGE_REF_N>`) and motion/style reference videos (`<VIDEO_REF_N>`).
- **Video Extensions & Turn-by-Turn Edits**:
  - Extend existing videos in 10-second increments up to 40 seconds total length while preserving character and audio consistency.
  - Perform natural-language video edits (e.g., style transfer, inpainting, outpainting) with state preservation via `previous_interaction_id`.
  - Smart audio control: preserve original audio or regenerate brand-new sound effects and music tracks from scratch (`--strip-audio`).
- **Batch Processing & Tooling**:
  - Complete CLI utilities for pre-processing videos with `ffmpeg` (`prep_video.py`), asset inspection (`inspect_video.py`), and parallel batch job runners.

---

### What We Can Build Together
- **Interactive Multimodal Assistants**: Voice agents that can see your screen or camera, inspect code, and speak back in real time.
- **AI Video Production Pipelines**: Automated text-to-video generators, short-form video editors, and looping visual synthesizers.
- **Autonomous Coding & Research Agents**: Cloud-hosted developer environments that clone Git repositories, execute tests, review code, or write exhaustive reports.
- **Real-Time Translation & Meeting Assistants**: Live transcription bots that translate live multilingual calls with smart formatting.

### Forking from an existing environment

If you've already set up a sandbox you like (installed packages, created files, etc.), you can fork it into a new agent using the `environment_id` from a previous interaction:

```python
my_forked_agent = client.agents.create(
    id="my-forked-agent",
    base_agent=AGENT,
    system_instruction="Your custom instructions here.",
    base_environment={"env_id": "YOUR_ENVIRONMENT_ID"},
)
```

This captures the exact state of that sandbox — all installed packages, files, and configuration.

In [26]:
import uuid # Ensure uuid is available if not in scope or kernel was restarted
my_forked_agent = client.agents.create(
    id=f"my-forked-agent-{uuid.uuid4().hex[:8]}",
    base_agent=AGENT,
    system_instruction="I want all your apps to use the Live API",
    base_environment={"env_id": interaction.environment_id},
)
print(f"✓ Agent forked: {my_forked_agent.id}")

✓ Agent forked: my-forked-agent-7cb816da


In [31]:
import uuid # Ensure uuid is available if not in scope or kernel was restarted

# The client.agents object does not have an 'update' method.
# To change an agent's properties, you typically need to create a new agent
# with the desired modifications.
# This will create a new agent instance with a unique ID, and update the
# `my_forked_agent` variable to refer to this new agent.

my_forked_agent = client.agents.create(
    id=f"my-forked-agent-updated-{uuid.uuid4().hex[:8]}", # Create a new agent with a new unique ID
    base_agent=AGENT,
    system_instruction="I want all your apps to use the Live API and be very friendly.",
    base_environment={"env_id": interaction.environment_id}, # Reusing the original environment ID
)
print(f"✓ Agent {my_forked_agent.id} created/replaced with new system instruction.")

✓ Agent my-forked-agent-updated-fe42fcb0 created/replaced with new system instruction.


### Managing agents (CRUD)

The `/agents` endpoint supports full lifecycle management:

In [34]:
# List all your agents.
print("Your agents:")
for agent in client.agents.list().agents:
    print(f"- {agent.id}")

# Get a specific agent's details.
agent = client.agents.get(id=f"my-data-analyst-{UNIQUE_SUFFIX}")
print(f"\nAgent details for {agent.id}:")
print(f"Base agent: {agent.base_agent}")
print(f"System instruction: {agent.system_instruction}")

Your agents:
- my-data-analyst-161cedde
- my-forked-agent
- my-forked-agent-161cedde
- my-forked-agent-7cb816da
- my-forked-agent-updated-9a4948dc
- my-forked-agent-updated-fe42fcb0
- my-gemini-api-agent-161cedde

Agent details for my-data-analyst-161cedde:
Base agent: antigravity-preview-05-2026
System instruction: You are a data analysis assistant. Always write Python code using pandas to answer questions. Show your code and output clearly. When creating visualizations, save them as PNG files.


In [35]:
# Clean up: delete the agents you created.
for agent_name in ["my-data-analyst", "my-forked-agent", "my-gemini-api-agent"]:
    try:
        client.agents.delete(id=agent_name)
        print(f"✓ Deleted {agent_name}")
    except Exception as e:
        print(f"  Failed to delete {agent_name}: {e}")

✓ Deleted my-data-analyst
✓ Deleted my-forked-agent
✓ Deleted my-gemini-api-agent


### Agent directory structure

Behind the API, an agent is defined by a simple set of files. This is what gets deployed when you create one:

```
my-agent/
├── agent.yaml       # Configuration: base agent, tools, environment
├── AGENTS.md        # System instructions (loaded automatically)
├── skills/          # Custom SKILL.md files that extend capabilities
└── workspace/       # Files seeded into the remote sandbox at startup
```

- **`agent.yaml`** maps directly to the `/agents` API resource
- **`AGENTS.md`** provides system instructions — automatically loaded by the harness
- **`skills/`** contains specialized `SKILL.md` files the agent discovers and uses
- **`workspace/`** files are injected into the sandbox at startup

This file-based structure makes agents easy to version-control, share, and iterate on. Check the [documentation](https://ai.google.dev/gemini-api/docs/custom-agents#file-based_customization) for more details.

## 6. Streaming

For longer tasks, enable streaming with `stream=True` to get real-time updates as the agent works. Instead of waiting for the complete response, you receive a stream of **Server-Sent Events (SSE)** that let you show progress to the user.

### Event types

The stream delivers events that tell you what the agent is doing:

| Event type | Meaning | What to do |
|------------|---------|------------|
| `interaction.created` | The interaction was created | Store the `id` for later reference |
| `interaction.status_update` | Status changed (e.g., `in_progress`) | Update UI status indicator |
| `step.start` | A new step began (thinking, tool call, output) | Show a loading indicator |
| `step.delta` | Incremental content — a chunk of text, thought, or tool output | **Append to display** — this is the main content |
| `step.stop` | A step completed | Hide loading indicator |
| `interaction.completed` | The agent finished all work | Finalize the UI |

The `step.delta` events are where the content lives. Each delta has a `type` (e.g., `text`, `thought`, `function_call`, `function_result`) and content you can render incrementally.

In [36]:
# Stream a response and collect the text as it arrives.
stream = client.interactions.create(
    agent=AGENT,
    input="Write a short poem about the ocean.",
    stream=True,
    environment="remote",
)

collected_text = []

for event in stream:
    # Show the event type so you can see the lifecycle.
    if event.event_type in ("interaction.created", "step.start", "step.stop", "interaction.completed"):
        print(f"[{event.event_type}]")

    # step.delta events carry the actual content.
    elif event.event_type == "step.delta":
        delta = event.delta
        if hasattr(delta, "text") and delta.text:
            print(delta.text, end="", flush=True)
            collected_text.append(delta.text)

print(f"\n\n--- Collected {len(collected_text)} text chunks ---")

[interaction.created]
[step.start]
[step.stop]
[step.start]
Beneath the wide and silver sky,  
The endless waters breathe and sigh.  
Tides of sapphire, crests of foam,  
A restless wanderer finding home.  

Depths of quiet, ancient grace,  
Shadows dancing across space;  
A timeless pulse against the shore,  
Whispering secrets forevermore.[step.stop]
[interaction.completed]


--- Collected 3 text chunks ---


## 7. Advanced features

### Network configuration

By default, the agent's sandbox has unrestricted outbound network access. You can control this with the `network` field:
- **Allowlist specific domains** — only requests to listed domains are permitted
- **Inject credentials** — automatically add headers (API keys, tokens) to outbound requests
- **Disable network** — set `network: "disabled"` to block all outbound traffic

In [37]:
# Allow the agent to call only the Gemini API, with an auto-injected API key.
interaction = client.interactions.create(
    agent=AGENT,
    input="Use curl to call the Gemini API and list available models. Show the first 3.",
    environment={
        "type": "remote",
        "network": {
            "allowlist": [
                {
                    "domain": "generativelanguage.googleapis.com",
                    "transform": [{"x-goog-api-key": GEMINI_API_KEY}],
                },
            ]
        },
    },
)

Markdown(interaction.output_text)

Here is the `curl` command used to query the Gemini API:

```bash
curl -s https://generativelanguage.googleapis.com/v1beta/models | jq '.models[:3]'
```

### Response (First 3 Models)

```json
[
  {
    "name": "models/gemini-2.5-flash",
    "version": "001",
    "displayName": "Gemini 2.5 Flash",
    "description": "Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.",
    "inputTokenLimit": 1048576,
    "outputTokenLimit": 65536,
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens",
      "createCachedContent",
      "batchGenerateContent"
    ],
    "temperature": 1,
    "topP": 0.95,
    "topK": 64,
    "maxTemperature": 2,
    "thinking": true
  },
  {
    "name": "models/gemini-2.5-pro",
    "version": "2.5",
    "displayName": "Gemini 2.5 Pro",
    "description": "Stable release (June 17th, 2025) of Gemini 2.5 Pro",
    "inputTokenLimit": 1048576,
    "outputTokenLimit": 65536,
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens",
      "createCachedContent",
      "batchGenerateContent"
    ],
    "temperature": 1,
    "topP": 0.95,
    "topK": 64,
    "maxTemperature": 2,
    "thinking": true
  },
  {
    "name": "models/gemini-2.5-flash-preview-tts",
    "version": "gemini-2.5-flash-exp-tts-2025-05-19",
    "displayName": "Gemini 2.5 Flash Preview TTS",
    "description": "Gemini 2.5 Flash Preview TTS",
    "inputTokenLimit": 8192,
    "outputTokenLimit": 16384,
    "supportedGenerationMethods": [
      "countTokens",
      "generateContent"
    ],
    "temperature": 1,
    "topP": 0.95,
    "topK": 64,
    "maxTemperature": 2
  }
]
```

### Download environment snapshots

You can download all the files the agent created or modified as a tar archive. This lets you retrieve the agent's work products — code, data, reports — from the sandbox.

In [38]:
import subprocess
import tarfile
import os

# Create an interaction where the agent produces files.
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Create a directory called 'project' with a README.md and a hello.py script. "
        "List the files you created."
    ),
    environment="remote",
)

env_id = interaction.environment_id
print(f"Environment ID: {env_id}")

Markdown(interaction.output_text)

Environment ID: ab0c9a4a34a00f56dc2023c5dcce3e72


I have created the directory `project` along with the requested files.

### Created Files:
- `project/`
  - `README.md`
  - `hello.py`

In [39]:
# Download the environment snapshot.
download_url = (
    f"https://generativelanguage.googleapis.com/v1beta/"
    f"files/environment-{env_id}:download?alt=media"
)

result = subprocess.run(
    ["curl", "-L", "-s", "-o", "snapshot.tar",
     "-H", f"x-goog-api-key: {GEMINI_API_KEY}",
     download_url],
    capture_output=True, text=True,
)

if os.path.exists("snapshot.tar") and os.path.getsize("snapshot.tar") > 0:
    with tarfile.open("snapshot.tar") as tar:
        print("Files in snapshot:")
        for member in tar.getmembers():
            print(f"  {member.name} ({member.size} bytes)")
else:
    print("Snapshot not available (environment may have expired).")

Files in snapshot:
  . (0 bytes)
  ./project (0 bytes)
  ./project/README.md (74 bytes)
  ./project/hello.py (78 bytes)


## Next steps

You've walked through the core capabilities of managed agents:

1. ✅ **Simple Q&A** — the agent can answer questions like an LLM
2. ✅ **Multi-turn** — persistent sandbox enables stateful conversations
3. ✅ **Built-in tools** — code execution, web search, file management
4. ✅ **Data loading** — inject files via inline, GCS, or GitHub sources
5. ✅ **Custom agents** — reusable configurations with instructions, skills, and environment
6. ✅ **Streaming** — real-time updates as the agent works
7. ✅ **Advanced** — network control and environment snapshots

### Learn more

- **[Getting Started notebook](./Get_started_interactions_api.ipynb)** — the `model=`-based Interactions API for standard generation, multi-turn, and tools
- **[Managed Agents documentation](https://ai.google.dev/gemini-api/docs/eap/gemini-agents/gemini-agents)** — full reference for the agent API
- **[Code Execution](./Code_Execution.ipynb)** — model-based code execution
- **[Search Grounding](./Search_Grounding.ipynb)** — model-based web search
- **[Function Calling](./Function_calling.ipynb)** — custom function declarations